<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB20_Case_Study_GMRT_Bathymetry_Interpolation_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# NB20 - Clase 20 -- Caso de estudio: batimetría GMRT -- prediciendo la profundidad del fondo marino a partir de sondeos escasos

## Bloque 4: Proyectos -- Casos de estudio (continuación)

`NB18` preguntaba si los campos de viento predicen la altura de ola. `NB19` preguntaba si la ruta de un barco revela su nacionalidad. Esta clase plantea una tercera pregunta real, estructuralmente distinta: **si solo dispones de un conjunto escaso de mediciones reales de profundidad, ¿con qué precisión puedes predecir la profundidad del fondo marino entre ellas?**

Esto no es una hipótesis. `Un levantamiento de sonar multihaz de cobertura completa de una zona costera es lento y caro` -- los buques navegan de un lado a otro en líneas solapadas, a pocos nudos, durante días o semanas por kilómetro cuadrado. Históricamente, y todavía hoy en muchas aguas poco levantadas, las cartas náuticas se construyen a partir de **sondeos escasos en líneas**, y la profundidad entre esas líneas hay que estimarla de algún modo. Esta clase trata ese problema de estimación como una tarea genuina de machine learning espacial, usando datos batimétricos reales de las aguas costeras alrededor de Cartagena -- el mismo litoral en el que se imparte este curso.


## Agenda (120 min)

| # | Sección | Minutos |
|---|---|---|
| 1 | Por qué importa la interpolación de sondeos escasos para el levantamiento hidrográfico | 5 |
| 2 | Datos reales: el servicio de batimetría GMRT | 5 |
| 3 | Descargar y analizar una rejilla batimétrica real | 10 |
| 4 | Explorar los datos reales: un mapa de la plataforma de Cartagena | 20 |
| 5 | Plantear la tarea: simular un levantamiento escaso | 5 |
| 6 | Regresión de k vecinos más próximos (técnica nueva) | 15 |
| 7 | Entrenar y evaluar kNN frente a Random Forest frente a una referencia ingenua | 30 |
| 8 | El experimento real: precisión frente a densidad del levantamiento | 20 |
| 9 | Interpretación honesta y limitaciones | 5 |
| 10 | Resumen de la clase, tarea, y qué sigue | 5 |

Como en cada clase de este curso, son guías aproximadas, no un guion -- si una sección se queda corta o larga, no pasa nada.


---

## 1. Por qué importa la interpolación de sondeos escasos


`La utilidad de una carta náutica depende por completo de cuán bien represente el fondo marino real` -- un barco, y especialmente un submarino o un buque de mucho calado, necesita saber dónde el agua es demasiado somera mucho antes de llegar allí. Producir ese conocimiento es caro. Un levantamiento moderno con ecosonda multihaz, realizado a una velocidad de levantamiento segura y con la cobertura de líneas solapadas necesaria para tener certeza de todo el fondo, puede llevarle a un buque de levantamiento días para cubrir unos pocos kilómetros cuadrados. El **estándar S-44 de la Organización Hidrográfica Internacional** fija requisitos mínimos de densidad de sondeo precisamente por este compromiso: los levantamientos más densos cuestan más, pero los más escasos corren el riesgo de pasar por alto un bajo fondo o un naufragio entre líneas de sondeo.

Históricamente -- y todavía hoy en grandes zonas de las aguas costeras del mundo que nunca se han vuelto a levantar por completo con equipos modernos -- las cartas se construyen a partir de **sondeos escasos, puntuales o en líneas**, con la profundidad *entre* ellos estimada por interpolación. Esta clase trata ese problema de estimación del mismo modo en que este curso ha tratado cualquier otro problema real: como algo sobre lo que se puede entrenar un modelo y evaluarlo honestamente, usando datos batimétricos reales como referencia (ground truth).

> **Para saber más**: [Batimetría (Wikipedia)](https://en.wikipedia.org/wiki/Bathymetry) | [Levantamiento hidrográfico (Wikipedia)](https://en.wikipedia.org/wiki/Hydrographic_survey) | [Estándar S-44 de la IHO, resumen (Wikipedia)](https://en.wikipedia.org/wiki/International_Hydrographic_Organization)


---

## 2. Datos reales: el servicio de batimetría GMRT


**GMRT (Global Multi-Resolution Topography)** es una síntesis global real, actualizada continuamente, de batimetría y topografía, mantenida por el Marine Geoscience Data System del Lamont-Doherty Earth Observatory (Universidad de Columbia). Combina datos reales de levantamientos de sonar multihaz, allí donde esos levantamientos se han aportado al archivo, con estimaciones de batimetría derivadas de altimetría por satélite que rellenan los huecos donde todavía no existe un levantamiento real.

**Esta es una advertencia honesta e importante que conviene tener presente el resto de la clase**: en las zonas que GMRT cubre solo con estimaciones derivadas de satélite (no con sondeos reales), la "referencia" (ground truth) con la que este notebook entrena y evalúa `es en sí misma la estimación de un modelo, no una medición real certificada`. Las zonas costeras cercanas a un puerto importante -- como la usada en esta clase -- tienen más probabilidades de contar con aportaciones de levantamientos reales que el océano abierto, pero esto no se puede verificar solo desde este notebook.

GMRT expone una API web **GridServer** gratuita y sin registro que devuelve una rejilla regional real para cualquier caja delimitadora (bounding box) que solicites -- sin clave de API, sin `getpass`, sin cuenta, a diferencia de todas las demás fuentes de datos de casos de estudio del Bloque 4 hasta ahora. Esta clase solicita una caja delimitadora real que cubre el litoral de Cartagena y la plataforma y el talud mediterráneos adyacentes, hasta profundidades reales de aproximadamente 2.500 m.

> **Para saber más**: [Sitio oficial de GMRT](https://www.gmrt.org/) | [Ryan et al. 2009, la síntesis GMRT (DOI)](https://doi.org/10.1029/2008GC002332) | [Ecosonda multihaz (Wikipedia)](https://en.wikipedia.org/wiki/Multibeam_echosounder)


---

## 3. Descargar y analizar una rejilla batimétrica real


La celda de abajo solicita una rejilla real al GridServer de GMRT, en formato **Esri ASCII grid** (`.asc`) -- un formato de texto simple y totalmente autodescriptivo: una cabecera de 6 líneas (tamaño de la rejilla, coordenadas de las esquinas, tamaño de celda, valor de "sin datos") seguida de los propios valores de elevación, una fila de la rejilla por línea de texto, leída de arriba abajo (de norte a sur).

No hace falta ni inicio de sesión ni clave, así que esta descarga o funciona directamente o falla con un error de red normal -- no hay ningún paso de credenciales en el que equivocarse.


In [ ]:
import urllib.request
import os

GMRT_URL = (
    "https://www.gmrt.org/services/GridServer"
    "?minlongitude=-1.6&maxlongitude=-0.6"
    "&minlatitude=37.3&maxlatitude=37.8"
    "&format=esriascii&resolution=low"
)

grid_path = "cartagena_bathymetry.asc"
if not os.path.exists(grid_path):
    urllib.request.urlretrieve(GMRT_URL, grid_path)

print(f"Downloaded {os.path.getsize(grid_path) / 1e6:.1f} MB to {grid_path}")


La siguiente celda analiza la cabecera **sin asumir sus valores exactos** -- el tamaño y la resolución de la rejilla pueden cambiar ligeramente entre solicitudes a medida que se actualiza la síntesis de GMRT, así que el código lee los valores de `ncols`/`nrows`/`cellsize`/esquinas que el fichero realmente indique, en lugar de fijar en el código los números vistos al construir este notebook.

In [ ]:
import numpy as np

with open(grid_path) as f:
    header = {}
    for _ in range(6):
        key, val = f.readline().split()
        header[key.lower()] = val
    ncols = int(header["ncols"])
    nrows = int(header["nrows"])
    xllcorner = float(header["xllcorner"])
    yllcorner = float(header["yllcorner"])
    cellsize = float(header["cellsize"])
    nodata_value = float(header["nodata_value"])
    grid = np.loadtxt(f)

assert grid.shape == (nrows, ncols), f"Unexpected grid shape {grid.shape}, expected ({nrows}, {ncols})"
n_nodata = (grid == nodata_value).sum()

print(f"Grid shape: {grid.shape} (rows x cols)")
print(f"Cell size: {cellsize:.6f} degrees (~{cellsize * 111_000:.0f} m at this latitude)")
print(f"No-data cells: {n_nodata} / {grid.size}")
print(f"Elevation range: {grid.min():.1f} m to {grid.max():.1f} m (negative = below sea level)")


Las rejillas Esri ASCII almacenan las filas **de norte a sur** (la primera fila del fichero es la más septentrional). La celda de abajo reconstruye la latitud y la longitud reales de cada celda solo a partir de la cabecera, y luego aplana la rejilla en un DataFrame ordenado -- el mismo patrón de "aplanar una rejilla espacial en filas" que usó `NB18` para sus rejillas de viento/oleaje de ERA5, aplicado aquí a una rejilla batimétrica en lugar de a una meteorológica.

In [ ]:
import pandas as pd

lons = xllcorner + cellsize * np.arange(ncols)
lats = yllcorner + cellsize * np.arange(nrows)[::-1]  # first row = northernmost

lon_grid, lat_grid = np.meshgrid(lons, lats)

bathy = pd.DataFrame({
    "longitude": lon_grid.ravel(),
    "latitude": lat_grid.ravel(),
    "elevation_m": grid.ravel(),
})
bathy = bathy[bathy["elevation_m"] != nodata_value].reset_index(drop=True)

print(f"{len(bathy):,} real grid cells with valid elevation data")
bathy.head()


---

## 4. Explorar los datos reales: un mapa de la plataforma de Cartagena


`elevation_m` es positivo en tierra y negativo bajo el agua -- `este único dataset real cubre tanto el litoral como el fondo marino`. La siguiente celda lo divide en `land` (elevación > 0) y `sea` (elevación <= 0, la batimetría real que interesa en esta clase) e informa de las proporciones reales y el rango de profundidad, antes de representar nada.

In [ ]:
sea = bathy[bathy["elevation_m"] <= 0].copy()
sea["depth_m"] = -sea["elevation_m"]
land = bathy[bathy["elevation_m"] > 0]

print(f"Sea cells: {len(sea):,} ({len(sea) / len(bathy):.1%})")
print(f"Land cells: {len(land):,} ({len(land) / len(bathy):.1%})")
print(f"Depth range: {sea['depth_m'].min():.1f} m to {sea['depth_m'].max():.1f} m")
print(sea["depth_m"].describe())


Siguiendo la misma regla que este curso ha usado desde el mapa de residuos espaciales de `NB18` -- cualquier dato real de lat/lon se representa sobre un mapa geográfico real con `cartopy`, no con un simple scatter plot de ejes sin etiquetar.

In [ ]:
%pip install -q cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(9, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
sc = ax.scatter(
    bathy["longitude"], bathy["latitude"], c=bathy["elevation_m"],
    cmap="terrain", s=2, transform=ccrs.PlateCarree(), vmin=-2500, vmax=900,
)
ax.coastlines(resolution="10m")
ax.add_feature(cfeature.BORDERS, linestyle=":")
gl = ax.gridlines(draw_labels=True, linestyle="--", alpha=0.4)
gl.top_labels = False
gl.right_labels = False
plt.colorbar(sc, label="Elevation (m) -- negative = depth below sea level", shrink=0.8)
plt.title("Real GMRT elevation/bathymetry -- Cartagena coast and Mediterranean shelf")
plt.show()


Un histograma hace concreta la distribución real de profundidad antes de representarla espacialmente:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(sea["depth_m"], bins=40, color="steelblue", edgecolor="white")
ax.set_xlabel("Depth (m)")
ax.set_ylabel("Number of grid cells")
ax.set_title("Real depth distribution -- Cartagena shelf and slope")
plt.show()


---

## 5. Plantear la tarea: simular un levantamiento escaso


La rejilla de GMRT de arriba se trata de aquí en adelante como **referencia real (ground truth)** -- cada celda de mar tiene un valor de profundidad real (o estimado por GMRT, según la advertencia de la Sección 2). La pregunta genuina es: si un buque de levantamiento solo hubiera muestreado una fracción pequeña y escasa de estos puntos -- como haría un levantamiento histórico real o limitado por presupuesto -- ¿con qué precisión se podría predecir el *resto* de las profundidades?

La celda de abajo simula esto tomando una pequeña muestra aleatoria de celdas de mar como los "sondeos conocidos" (el conjunto de entrenamiento) y reservando el conjunto restante, mucho más grande, de celdas de mar como puntos genuinamente no vistos cuya profundidad real el modelo nunca llega a usar para entrenar -- evaluados solo después, la misma disciplina que toda reserva genuina en este curso desde `NB07`.

In [ ]:
from sklearn.model_selection import train_test_split

SURVEY_FRACTION = 0.02  # 2% of sea cells "surveyed" -- a deliberately sparse starting point

known, unknown = train_test_split(sea, train_size=SURVEY_FRACTION, random_state=42)

X_known, y_known = known[["longitude", "latitude"]], known["depth_m"]
X_unknown, y_unknown = unknown[["longitude", "latitude"]], unknown["depth_m"]

print(f"Simulated survey: {len(known):,} known soundings ({SURVEY_FRACTION:.1%} of all sea cells)")
print(f"Held out to test against: {len(unknown):,} real depth values the model never trains on")


---

## 6. Regresión de k vecinos más próximos (técnica nueva)


Todos los modelos que este curso ha usado hasta ahora -- regresión lineal/logística, árboles de decisión y sus ensembles, SVM, redes neuronales -- aprenden una función explícita a partir de los datos de entrenamiento y luego aplican esa función a los puntos nuevos. La **regresión de k vecinos más próximos (kNN)** funciona de forma completamente distinta: no "aprende" nada en ese sentido. Para predecir el valor de un punto nuevo, simplemente busca los `k` puntos de entrenamiento más cercanos (aquí, por distancia geográfica real) y promedia sus valores conocidos.

Este es un encaje conceptual inusualmente bueno específicamente para la interpolación espacial: dos puntos cercanos entre sí en el fondo marino normalmente *sí* tienen profundidades parecidas (el fondo marino es en su mayoría una superficie suave, solo raramente alterada por acantilados, naufragios o arrecifes) -- `exactamente la suposición que kNN hace explícita, y exactamente en lo que también se apoya implícitamente quien elabora una carta al interpolar entre líneas de sondeo`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(7)
neighbors_x = rng.uniform(0, 10, 25)
neighbors_y = rng.uniform(0, 10, 25)
query_x, query_y = 5.0, 5.0

dist = np.hypot(neighbors_x - query_x, neighbors_y - query_y)
k = 5
nearest = np.argsort(dist)[:k]

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(neighbors_x, neighbors_y, c="steelblue", label="known soundings")
ax.scatter(neighbors_x[nearest], neighbors_y[nearest], c="orange", s=90,
           edgecolor="black", label=f"{k} nearest neighbors", zorder=3)
ax.scatter([query_x], [query_y], c="red", marker="*", s=250,
           edgecolor="black", label="query point (unknown depth)", zorder=4)
radius = dist[nearest].max()
circle = plt.Circle((query_x, query_y), radius, fill=False, linestyle="--", color="orange")
ax.add_patch(circle)
ax.set_aspect("equal")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
ax.set_title(f"k-Nearest Neighbors (k={k}): predict from the {k} closest known points")
plt.tight_layout()
plt.show()


Para regresión, la predicción de kNN es el **promedio** (opcionalmente ponderado por distancia, para que los puntos más cercanos cuenten más) de los valores conocidos de los `k` vecinos. El propio `k` es un hiperparámetro con un compromiso real, igual que la profundidad del árbol en `NB08`: un `k` pequeño sigue de cerca el detalle local pero es sensible al ruido de sondeos individuales; un `k` grande suaviza rasgos locales reales (un afloramiento rocoso, una fosa) que quien elabora una carta necesita realmente ver.

> **Para saber más**: [Algoritmo de los k vecinos más próximos (Wikipedia)](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm) | [documentación de `KNeighborsRegressor` de scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html) | [Interpolación espacial / kriging (Wikipedia)](https://en.wikipedia.org/wiki/Kriging)

---

## 7. Entrenar y evaluar kNN frente a Random Forest frente a una referencia ingenua


Se comparan tres predictores sobre el mismo levantamiento escaso de la Sección 5, todos entrenados solo con `X_known`/`y_known` y evaluados solo sobre el conjunto reservado `X_unknown`/`y_unknown`:

- Una **referencia ingenua (baseline)**: predecir la profundidad de cada punto desconocido como la profundidad exacta de su único sondeo conocido más cercano (`k=1`, sin ponderar) -- lo más simple que podría hacer a mano quien elabora una carta, y el listón que cualquier modelo real necesita superar.
- **Regresión kNN** (`k=8`, ponderada por distancia) -- la técnica nueva de esta clase.
- **Regresión con Random Forest** -- ya conocida de `NB07`/`NB10`/`NB18`, incluida para que la nueva técnica específicamente espacial se juzgue frente a un modelo genérico sólido, y no solo frente a la referencia ingenua.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

models = {
    "Naive nearest-sounding": KNeighborsRegressor(n_neighbors=1),
    "kNN (k=8, distance-weighted)": KNeighborsRegressor(n_neighbors=8, weights="distance"),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
}

results = {}
predictions = {}
for name, model in models.items():
    model.fit(X_known, y_known)
    pred = model.predict(X_unknown)
    predictions[name] = pred
    mae = mean_absolute_error(y_unknown, pred)
    rmse = mean_squared_error(y_unknown, pred) ** 0.5  # sqrt(MSE): `squared=False` was removed in newer scikit-learn
    results[name] = {"MAE (m)": mae, "RMSE (m)": rmse}

results_df = pd.DataFrame(results).T
print(f"Evaluated on {len(y_unknown):,} real held-out depth soundings (survey fraction {SURVEY_FRACTION:.1%})")
results_df


Un diagrama de barras hace que la comparación entre los tres sea más fácil de leer que solo la tabla:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, metric in zip(axes, ["MAE (m)", "RMSE (m)"]):
    ax.bar(results_df.index, results_df[metric], color=["steelblue", "darkorange", "seagreen"])
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


Una única cifra de error esconde *dónde* se equivoca un modelo. El mapa de abajo representa el error real de predicción (profundidad predicha menos profundidad real) del modelo con mejor desempeño en cada punto reservado -- un patrón sistemático (errores agrupados cerca de la costa, o cerca de los puntos más profundos) diría algo real sobre dónde es fiable la interpolación con levantamiento escaso y dónde no lo es; un error disperso y sin estructura sugeriría que el modelo está rindiendo tan bien como permiten los datos escasos.

In [ ]:
best_name = results_df["RMSE (m)"].idxmin()
error = predictions[best_name] - y_unknown.values
print(f"Best model on this run: {best_name}")

fig = plt.figure(figsize=(9, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
sc = ax.scatter(
    X_unknown["longitude"], X_unknown["latitude"], c=error,
    cmap="coolwarm", s=3, vmin=-100, vmax=100, transform=ccrs.PlateCarree(),
)
ax.scatter(X_known["longitude"], X_known["latitude"], c="black", s=4,
           marker="x", transform=ccrs.PlateCarree(), label="known soundings")
ax.coastlines(resolution="10m")
gl = ax.gridlines(draw_labels=True, linestyle="--", alpha=0.4)
gl.top_labels = False
gl.right_labels = False
plt.colorbar(sc, label="Prediction error (m): predicted - true depth", shrink=0.8)
ax.legend(loc="lower left")
plt.title(f"{best_name}: real prediction error at each held-out point")
plt.show()


**Pruébalo tú mismo**: un scatter de predicho frente a real para el mejor modelo es una vista distinta y complementaria al mapa de error espacial de arriba — ¿se pegan los puntos de cerca a la diagonal, o hay un sesgo sistemático (p. ej., infrapredecir la profundidad de forma constante)?

In [ ]:
best_pred = predictions[best_name]

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_unknown, best_pred, alpha=0.3, s=5)
lims = [y_unknown.min(), y_unknown.max()]
ax.plot(lims, lims, "r--", label="Perfect prediction")
ax.set_xlabel("Actual depth (m)")
ax.set_ylabel("Predicted depth (m)")
ax.set_title(f"{best_name}: predicted vs. actual depth")
ax.legend()
plt.show()


---

## 8. El experimento real: precisión frente a densidad del levantamiento


La Sección 5 fijó la densidad del levantamiento en un único 2%, elegido de forma arbitraria. La pregunta genuinamente útil para planificar un levantamiento real es distinta: **¿cuánto mejora realmente la precisión a medida que se levanta más fondo marino?** La celda de abajo repite exactamente el mismo procedimiento de entrenar/evaluar de la Sección 7 -- solo con kNN, para que este bucle sea rápido -- en varios niveles reales de fracción de levantamiento, y representa el RMSE frente a la densidad del levantamiento.

In [ ]:
fractions = [0.002, 0.005, 0.01, 0.02, 0.05, 0.10]
density_results = []

for frac in fractions:
    known_f, unknown_f = train_test_split(sea, train_size=frac, random_state=42)
    knn = KNeighborsRegressor(n_neighbors=8, weights="distance")
    knn.fit(known_f[["longitude", "latitude"]], known_f["depth_m"])
    pred_f = knn.predict(unknown_f[["longitude", "latitude"]])
    rmse_f = mean_squared_error(unknown_f["depth_m"], pred_f) ** 0.5  # sqrt(MSE): `squared=False` was removed in newer scikit-learn
    density_results.append({"survey_fraction": frac, "n_known": len(known_f), "RMSE_m": rmse_f})
    print(f"Survey fraction {frac:>6.1%}  ({len(known_f):>5,} soundings)  ->  RMSE = {rmse_f:6.2f} m")

density_df = pd.DataFrame(density_results)


Los números impresos arriba ya muestran la tendencia; el gráfico de abajo hace que la forma del compromiso (¿mejora la precisión de manera constante, o hay un punto de rendimientos decrecientes?) sea más fácil de leer de un vistazo.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(density_df["survey_fraction"] * 100, density_df["RMSE_m"], marker="o")
ax.set_xlabel("Survey coverage (% of sea cells actually sounded)")
ax.set_ylabel("kNN RMSE on held-out depth (m)")
ax.set_title("Real trade-off: sounding density vs. interpolation accuracy")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Pruébalo tú mismo**: cuantifica lo que muestra el gráfico — la mejora real de RMSE que se gana por cada 1% adicional de cobertura del levantamiento, en cada paso. ¿Dónde empieza a reducirse claramente el beneficio de levantar más?

In [ ]:
density_df["RMSE_improvement_m"] = -density_df["RMSE_m"].diff()
density_df["coverage_increase_pct"] = density_df["survey_fraction"].diff() * 100
density_df["improvement_per_1pct_coverage"] = density_df["RMSE_improvement_m"] / density_df["coverage_increase_pct"]
density_df.round(3)


---

## 9. Interpretación honesta y limitaciones


Merece la pena exponer con claridad, sin esconderlas, algunas limitaciones reales de este ejercicio:

- **La simulación de escasez aleatoria no es cómo se planifican los levantamientos reales.** Un levantamiento hidrográfico real sigue líneas de tránsito planificadas, no una dispersión de puntos uniformemente aleatoria -- el muestreo aleatorio es lo más fácil de simular aquí, pero es un caso genuinamente más favorable que un levantamiento real en líneas, que deja huecos largos y sistemáticos entre líneas en lugar de huecos pequeños por todas partes.
- **La propia "referencia real" (ground truth) tiene la advertencia planteada en la Sección 2.** Donde la cobertura de GMRT en esta zona procede de datos reales de multihaz, esta evaluación se hace frente a la profundidad real del fondo marino. Donde procede de la propia estimación de GMRT derivada de satélite, este notebook en realidad está probando "qué tan bien reproduce kNN la estimación de otro modelo" -- una afirmación distinta y más débil. Esto no se puede resolver sin una fuente independiente y certificada para esta zona exacta.
- **La interpolación a partir de puntos escasos, por bueno que parezca el RMSE, no puede recuperar rasgos más pequeños que el propio espaciado del levantamiento** -- un naufragio, un pináculo rocoso aislado, o un canal estrecho simplemente no aparecerán si ningún sondeo cayó jamás sobre ellos. Un error medio bajo no significa que una carta interpolada sea segura para navegar por ella; la práctica hidrográfica real exige o bien una cobertura suficientemente densa, o bien un margen de incertidumbre/peligro documentado, exactamente el tipo de cifra que la curva de RMSE de este notebook podría informar pero no sustituir.

Nada de esto hace que el experimento de densidad de levantamiento de arriba carezca de sentido -- es una relación real, evaluada honestamente, entre la densidad de sondeo y la precisión alcanzable sobre un terreno real. Simplemente no es un sustituto de un estándar hidrográfico real.

---

## Resumen de la clase

- Batimetría real de la costa de Cartagena, descargada en directo y sin inicio de sesión desde el GridServer público de GMRT -- la primera fuente de datos del Bloque 4 para la que este curso no necesitó ninguna credencial.
- Se planteó un compromiso real de coste/precisión propio de la ingeniería naval (la densidad del levantamiento hidrográfico) como una tarea genuina de machine learning espacial, evaluada honestamente.
- **Regresión de k vecinos más próximos**, una técnica nueva para este curso, introducida específicamente porque su suposición de que "los puntos cercanos se parecen" encaja de forma natural con la interpolación espacial.
- Se comparó kNN frente a una referencia ingenua y frente al Random Forest, ya familiar por `NB07`/`NB10`/`NB18`, sobre un conjunto reservado genuino de valores reales de profundidad que el modelo nunca usó para entrenar.
- El experimento real de densidad de levantamiento (Sección 8) es el tipo de cifra concreta que podría usar una decisión real de planificación de un levantamiento -- y sus propias limitaciones honestas (Sección 9) forman parte del resultado tanto como la propia curva de RMSE.

## Para la próxima clase

Otro caso de estudio del Bloque 4 -- tema por confirmar, continuando el mismo patrón: un dataset real, una pregunta real, y una reserva genuina allí donde la pregunta lo requiera.

## Tarea / Ideas de práctica

1. Repite el experimento de densidad de la Sección 8 con Random Forest en lugar de kNN -- ¿se mantiene la misma relación entre precisión y densidad, o un método necesita menos datos que el otro para alcanzar un RMSE dado?
2. Cambia `n_neighbors` en el modelo kNN de la Sección 7 (prueba 1, 3, 8, 30) con una fracción de levantamiento fija -- representa el RMSE frente a `k` y conecta el resultado con el compromiso detalle-local-frente-a-suavizado de la Sección 6.
3. En lugar de un "levantamiento" uniformemente aleatorio, simula un patrón real de levantamiento en líneas (p. ej., conserva solo los puntos dentro de una distancia fija de unas pocas líneas rectas norte-sur) con el mismo número total de puntos que una de las fracciones de la Sección 8 -- ¿empeora la precisión de la interpolación, en línea con la advertencia de la Sección 9 sobre que los levantamientos en líneas son más difíciles que el muestreo aleatorio?
4. Solicita una rejilla de GMRT para una zona costera real distinta (cambia la caja delimitadora de la Sección 3) y vuelve a ejecutar el notebook completo -- ¿se parece la curva de precisión frente a densidad, o cambia la relación el terreno local del fondo marino (un borde de plataforma abrupto frente a una pendiente suave)?
5. Vuelve a añadir los puntos de `elevation_m > 0` (tierra) a una tarea de interpolación combinada -- ¿ayuda o perjudica mezclar puntos de tierra y de mar a la hora de predecir la profundidad cerca del litoral, donde se encuentran ambos regímenes?

> ***Como siempre: un dataset real con una limitación expuesta con honestidad enseña más que uno limpio que esconde sus suposiciones.***